# 8. N8N ORCHESTRATION PREPARATION
## Daily Customer Churn Predictor · VivaMarket Brasil

---

**INPUT:** `../data/processed/churn_predictions_YYYYMMDD.parquet`, `../data/processed/churn_explainability_YYYYMMDD.parquet`, and `../models/churn_scoring_package_YYYYMMDD.joblib`

**OUTPUT:** `../data/processed/retention_actions_YYYYMMDD.parquet`, `../n8n/daily_churn_retention_workflow_YYYYMMDD.json`, and `../reports/n8n_orchestration_YYYYMMDD.html`

*A production-minded retention-action payload and an n8n workflow blueprint aligned with the project retention strategy.*


---
## 8.1. STARTING SITUATION


The project now has risk scores, explainability outputs and a deployment-ready scoring package. The remaining operational step is to define exactly what the daily orchestration should send downstream: who should be contacted, through which channels, with what incentive, and under which guardrails.


---
## 8.2. NOTEBOOK OBJECTIVE


- **Business objective:** convert scored customers into daily retention actions that match the High / Medium / Low framework from the retention strategy document.
- **Technical objective:** build a workflow-ready payload and a concrete n8n JSON blueprint that can later be implemented with minimal ambiguity.


In [1]:
import json
import logging
from datetime import datetime
from pathlib import Path
from zoneinfo import ZoneInfo

import pandas as pd

logging.basicConfig(level=logging.INFO, format='%(asctime)s | %(levelname)s | %(message)s', force=True)
logger = logging.getLogger('nb08_n8n_orchestration')
logger.info('NB08 started: n8n orchestration preparation.')


2026-05-02 00:38:37,053 | INFO | NB08 started: n8n orchestration preparation.


In [2]:
PROJECT_ROOT = Path.cwd().resolve().parent
PROCESSED_DIR = PROJECT_ROOT / 'data' / 'processed'
REPORTS_DIR = PROJECT_ROOT / 'reports'
N8N_DIR = PROJECT_ROOT / 'n8n'
REPORTS_DIR.mkdir(parents=True, exist_ok=True)
N8N_DIR.mkdir(parents=True, exist_ok=True)

run_date_tag = datetime.now(ZoneInfo('Europe/Paris')).strftime('%Y%m%d')
prediction_path = sorted(PROCESSED_DIR.glob('churn_predictions_*.parquet'))[-1]
explainability_path = sorted(PROCESSED_DIR.glob('churn_explainability_*.parquet'))[-1]
actions_path = PROCESSED_DIR / f'retention_actions_{run_date_tag}.parquet'
workflow_json_path = N8N_DIR / f'daily_churn_retention_workflow_{run_date_tag}.json'
orchestration_html_path = REPORTS_DIR / f'n8n_orchestration_{run_date_tag}.html'


In [3]:
prediction_df = pd.read_parquet(prediction_path)
explainability_df = pd.read_parquet(explainability_path)
action_df = prediction_df.merge(
    explainability_df[[
        'customer_unique_id', 'snapshot_key', 'top_driver_group', 'recommended_offer_type',
        'recommended_discount_pct', 'free_shipping_flag', 'vip_human_touch_flag', 'ltv_segment'
    ]],
    on=['customer_unique_id', 'snapshot_key'],
    how='left',
    validate='one_to_one',
)

action_df['top_driver_group'] = action_df['top_driver_group'].fillna('unassigned_sample_gap')
action_df['recommended_offer_type'] = action_df['recommended_offer_type'].fillna(
    action_df['risk_tier'].map({
        'HIGH': 'reactivation_urgency',
        'MEDIUM': 'repeat_purchase_nurturing',
        'LOW': 'loyalty_fidelization',
    })
)
action_df['recommended_discount_pct'] = action_df['recommended_discount_pct'].fillna(
    action_df['risk_tier'].map({'HIGH': 25, 'MEDIUM': 12, 'LOW': 0})
)
action_df['free_shipping_flag'] = action_df['free_shipping_flag'].fillna(
    action_df['risk_tier'].isin(['HIGH', 'MEDIUM'])
)
action_df['vip_human_touch_flag'] = action_df['vip_human_touch_flag'].fillna(False)
action_df['ltv_segment'] = action_df['ltv_segment'].astype(object).fillna('UNASSIGNED')
action_df['primary_channels'] = action_df['risk_tier'].map({
    'HIGH': 'email,push,sms',
    'MEDIUM': 'email,push',
    'LOW': 'email,in_app',
})
action_df['contact_policy'] = action_df['risk_tier'].map({
    'HIGH': 'day0_email_push__day3_sms_if_no_open__day7_last_call',
    'MEDIUM': 'every_3_to_7_days_nurturing',
    'LOW': 'weekly_loyalty_or_value_content',
})
action_df['control_group_flag'] = False
high_risk_idx = action_df[action_df['risk_tier'] == 'HIGH'].sample(frac=0.15, random_state=42).index
action_df.loc[high_risk_idx, 'control_group_flag'] = True
action_df['send_action_flag'] = ~action_df['control_group_flag']
action_df['offer_code_stub'] = action_df.apply(lambda row: f"{row['risk_tier'][:1]}-{row['snapshot_key']}-{row.name}", axis=1)
action_df.to_parquet(actions_path, index=False)
logger.info('Retention actions parquet saved to %s', actions_path)
action_df.head()


2026-05-02 00:38:37,510 | INFO | Retention actions parquet saved to /data/.openclaw/workspace/projects/TFM/daily-customer-churn-predictor/data/processed/retention_actions_20260502.parquet


,customer_unique_id,snapshot_key,snapshot_date,recency_days,total_orders,total_payment_value,orders_30d,orders_90d,churn_90d_label,churn_probability,...,recommended_offer_type,recommended_discount_pct,free_shipping_flag,vip_human_touch_flag,ltv_segment,primary_channels,contact_policy,control_group_flag,send_action_flag,offer_code_stub
0,0004bd2a26a76fe21f786e4fbd80607f,20180501,2018-05-01,26,1,166.98,1.0,1,1,0.558454,...,repeat_purchase_nurturing,12.0,True,False,UNASSIGNED,"email,push",every_3_to_7_days_nurturing,False,True,M-20180501-0
1,00050ab1314c0e55a6ca13cf7181fecf,20180501,2018-05-01,11,1,35.38,1.0,1,1,0.572675,...,repeat_purchase_nurturing,12.0,True,False,UNASSIGNED,"email,push",every_3_to_7_days_nurturing,False,True,M-20180501-1
2,00053a61a98854899e70ed204dd4bafe,20180501,2018-05-01,62,1,419.18,0.0,1,1,0.896241,...,reactivation_urgency,25.0,True,False,UNASSIGNED,"email,push,sms",day0_email_push__day3_sms_if_no_open__day7_las...,False,True,H-20180501-2
3,0005ef4cd20d2893f0d9fbd94d3c0d97,20180501,2018-05-01,50,1,129.76,0.0,1,1,0.770620,...,reactivation_urgency,25.0,True,False,UNASSIGNED,"email,push,sms",day0_email_push__day3_sms_if_no_open__day7_las...,False,True,H-20180501-3
4,00090324bbad0e9342388303bb71ba0a,20180501,2018-05-01,38,1,63.66,0.0,1,1,0.708131,...,reactivation_urgency,25.0,True,False,UNASSIGNED,"email,push,sms",day0_email_push__day3_sms_if_no_open__day7_las...,False,True,H-20180501-4


In [4]:
workflow_definition = {
    'name': 'Daily Churn Retention Actions',
    'schedule': '0 2 * * * America/Sao_Paulo',
    'description': 'Daily orchestration blueprint for VivaMarket Brasil churn retention actions.',
    'nodes': [
        {'id': 1, 'name': 'Cron Trigger', 'type': 'n8n-nodes-base.cron'},
        {'id': 2, 'name': 'Read Predictions', 'type': 'n8n-nodes-base.readBinaryFile'},
        {'id': 3, 'name': 'Merge Explainability', 'type': 'n8n-nodes-base.merge'},
        {'id': 4, 'name': 'Risk Switch', 'type': 'n8n-nodes-base.switch'},
        {'id': 5, 'name': 'Generate Coupon', 'type': 'n8n-nodes-base.httpRequest'},
        {'id': 6, 'name': 'Send Email', 'type': 'n8n-nodes-base.sendGrid'},
        {'id': 7, 'name': 'Send Push', 'type': 'n8n-nodes-base.oneSignal'},
        {'id': 8, 'name': 'Send SMS', 'type': 'n8n-nodes-base.twilio'},
        {'id': 9, 'name': 'Log Actions', 'type': 'n8n-nodes-base.postgres'},
        {'id': 10, 'name': 'Error Handler', 'type': 'n8n-nodes-base.emailSend'},
    ],
    'policy': {
        'max_contacts_30d': 4,
        'sms_only_if_email_not_opened': True,
        'control_group_high_risk_share': 0.15,
        'time_windows_local': {'email': '09:00-20:00', 'sms': '10:00-19:00'},
    },
}
workflow_json_path.write_text(json.dumps(workflow_definition, indent=2), encoding='utf-8')
logger.info('Workflow JSON saved to %s', workflow_json_path)


2026-05-02 00:38:37,529 | INFO | Workflow JSON saved to /data/.openclaw/workspace/projects/TFM/daily-customer-churn-predictor/n8n/daily_churn_retention_workflow_20260502.json


In [5]:
orchestration_summary = (
    action_df.groupby(['risk_tier', 'recommended_offer_type', 'primary_channels'], observed=False)
    .agg(
        rows_n=('customer_unique_id', 'size'),
        send_action_rows=('send_action_flag', 'sum'),
        control_rows=('control_group_flag', 'sum'),
        avg_discount_pct=('recommended_discount_pct', 'mean'),
    )
    .reset_index()
    .sort_values(['risk_tier', 'rows_n'], ascending=[True, False])
)

html_parts = [
    '<html><head><meta charset="utf-8"><title>N8N Orchestration</title></head><body>',
    '<h1>N8N ORCHESTRATION BLUEPRINT</h1>',
    '<h2>Workflow definition</h2>', f'<pre>{json.dumps(workflow_definition, indent=2)}</pre>',
    '<h2>Action summary</h2>', orchestration_summary.to_html(index=False),
    '<h2>Sample payload</h2>', action_df.head(25).to_html(index=False),
    '</body></html>'
]
orchestration_html_path.write_text('\n'.join(html_parts), encoding='utf-8')
logger.info('Orchestration report saved to %s', orchestration_html_path)
orchestration_summary.head(12)


2026-05-02 00:38:37,555 | INFO | Orchestration report saved to /data/.openclaw/workspace/projects/TFM/daily-customer-churn-predictor/reports/n8n_orchestration_20260502.html


,risk_tier,recommended_offer_type,primary_channels,rows_n,send_action_rows,control_rows,avg_discount_pct
6,LOW,loyalty_fidelization,"email,in_app",3234,3234,0,0.0
15,LOW,value_bundle_offer,"email,in_app",239,239,0,0.0
9,LOW,reactivation_urgency,"email,in_app",50,50,0,0.0
3,LOW,logistics_apology_priority_shipping,"email,in_app",1,1,0,0.0
0,LOW,category_specific_winback,"email,in_app",0,0,0,NaN
1,LOW,category_specific_winback,"email,push",0,0,0,NaN
2,LOW,category_specific_winback,"email,push,sms",0,0,0,NaN
4,LOW,logistics_apology_priority_shipping,"email,push",0,0,0,NaN
5,LOW,logistics_apology_priority_shipping,"email,push,sms",0,0,0,NaN
7,LOW,loyalty_fidelization,"email,push",0,0,0,NaN


---
## 8.3. NOTEBOOK CLOSURE


The orchestration stage now has a concrete action payload, an explicit control-group policy, and a daily n8n workflow blueprint aligned with the retention strategy. That means the project can move from model outputs to campaign operations without reinterpreting business rules every day.

The final notebook should consolidate these artifacts into a reporting view that helps monitor quality, campaign mix and future model drift.
